# Notebook 7: Full Architecture -- Everything Together

**What you'll learn:**
- How all the pieces fit together (model + tools + hooks + conversation manager)
- Building a fully configured agent from scratch
- Multi-turn conversation with tools and hooks
- Full diagnostic inspection of the agent
- Complete request flow trace

**Prerequisite:** Complete all previous notebooks (NB1-NB6)

**Companion reading:** `07-architecture-map.md`

---
## The Full Architecture

Here's how all the pieces connect:

```
                          +-------------------+
                          |      Agent        |
                          |                   |
  You ---> agent("...") ->|  system_prompt    |
                          |  messages []      |
                          |  state {}         |
                          +---+-----+-----+---+
                              |     |     |
                   +----------+     |     +----------+
                   |                |                |
              +----v----+    +------v------+   +----v----+
              |  Model  |    | ToolRegistry|   |  Hooks  |
              |         |    |             |   |         |
              |Bedrock  |    | get_weather |   | Logging |
              |Anthropic|    | calculator  |   | Guard   |
              |OpenAI   |    | save_note   |   | Retry   |
              +---------+    +-------------+   +---------+
                   |                |                |
                   +--------+-------+--------+-------+
                            |                |
                    +-------v--------+  +----v---------+
                    |  Event Loop    |  | Conversation |
                    |                |  |   Manager    |
                    | model call     |  |              |
                    | tool exec      |  | sliding      |
                    | recurse        |  | window       |
                    +----------------+  +--------------+
```

### Dependency Graph

```
Agent
  |-- Model (brain -- calls the AI)
  |-- ToolRegistry (catalog of tools)
  |     |-- AgentTool (each registered tool)
  |-- ToolExecutor (runs tools, default: thread pool)
  |-- HookRegistry (lifecycle callbacks)
  |     |-- HookProvider (each registered hook)
  |-- ConversationManager (manages memory)
  |-- EventLoop (the core reasoning engine)
  |     |-- stream_messages() (calls model)
  |     |-- _handle_model_execution() (model phase)
  |     |-- _handle_tool_execution() (tool phase)
  |     |-- recurse_event_loop() (loops back)
  |-- AgentResult (what you get back)
```

In [ ]:
# ============================================================
# STEP 1: Import everything we need
# ============================================================

import json
import time

# Core SDK
from strands import Agent, tool

# Model
from strands.models.bedrock import BedrockModel

# Hooks
from strands.hooks.registry import HookRegistry, HookProvider
from strands.hooks.events import (
    BeforeInvocationEvent,
    AfterInvocationEvent,
    BeforeModelCallEvent,
    AfterModelCallEvent,
    BeforeToolCallEvent,
    AfterToolCallEvent,
)

# Conversation manager
from strands.agent.conversation_manager.sliding_window import SlidingWindowConversationManager

# Tool context
from strands.types.tools import ToolContext

print("All imports successful.")

In [ ]:
# ============================================================
# STEP 2: Define 3 tools
# ============================================================

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city.
    
    Args:
        city: The name of the city.
    """
    # Simulated weather data
    weather_data = {
        "seattle": "62F, cloudy",
        "new york": "75F, sunny",
        "london": "55F, rainy",
    }
    return weather_data.get(city.lower(), f"72F, pleasant in {city}")

@tool
def calculator(expression: str) -> str:
    """Perform a math calculation.
    
    Args:
        expression: A math expression like '2 + 2' or '15 * 7'.
    """
    return str(eval(expression))

@tool
def save_note(title: str, content: str, tool_context: ToolContext) -> str:
    """Save a note for later reference.
    
    Args:
        title: Title of the note.
        content: Content to save.
    """
    # Uses ToolContext to access agent state
    notes = tool_context.agent.state.setdefault("notes", [])
    notes.append({"title": title, "content": content})
    return f"Saved note '{title}' (total notes: {len(notes)})"

print("3 tools defined: get_weather, calculator, save_note")

In [ ]:
# ============================================================
# STEP 3: Define hooks
# ============================================================

class TimelineHook(HookProvider):
    """Records a timeline of all events for later inspection."""
    
    def __init__(self):
        self.timeline = []  # Stores (timestamp, event_name, details)
        self.start_time = None
    
    def register_hooks(self, registry: HookRegistry, **kwargs):
        registry.add_callback(BeforeInvocationEvent, self.on_before_invocation)
        registry.add_callback(AfterInvocationEvent, self.on_after_invocation)
        registry.add_callback(BeforeModelCallEvent, self.on_before_model)
        registry.add_callback(AfterModelCallEvent, self.on_after_model)
        registry.add_callback(BeforeToolCallEvent, self.on_before_tool)
        registry.add_callback(AfterToolCallEvent, self.on_after_tool)
    
    def _log(self, name, details=""):
        elapsed = time.time() - self.start_time if self.start_time else 0
        self.timeline.append((elapsed, name, details))
    
    def on_before_invocation(self, event, **kwargs):
        self.start_time = time.time()
        self._log("BeforeInvocation", "Agent starting")
    
    def on_after_invocation(self, event, **kwargs):
        self._log("AfterInvocation", "Agent done")
    
    def on_before_model(self, event, **kwargs):
        self._log("BeforeModelCall", "Calling AI model")
    
    def on_after_model(self, event, **kwargs):
        self._log("AfterModelCall", f"stop_reason={event.stop_reason}")
    
    def on_before_tool(self, event, **kwargs):
        self._log("BeforeToolCall", f"tool={event.tool_name}")
    
    def on_after_tool(self, event, **kwargs):
        self._log("AfterToolCall", f"tool={event.tool_name}")
    
    def print_timeline(self):
        """Print the recorded timeline."""
        print("=== Event Timeline ===")
        for elapsed, name, details in self.timeline:
            print(f"  [{elapsed:6.2f}s] {name:25s} | {details}")
        print()

class SafetyGuard(HookProvider):
    """Blocks tool calls to dangerous tools."""
    
    def __init__(self, blocked_tools=None):
        self.blocked_tools = blocked_tools or []
    
    def register_hooks(self, registry: HookRegistry, **kwargs):
        registry.add_callback(BeforeToolCallEvent, self.on_before_tool)
    
    def on_before_tool(self, event: BeforeToolCallEvent, **kwargs):
        if event.tool_name in self.blocked_tools:
            event.cancel_tool(f"Tool '{event.tool_name}' blocked by safety policy.")

print("2 hooks defined: TimelineHook, SafetyGuard")

In [ ]:
# ============================================================
# STEP 4: Create a fully configured agent
# ============================================================

# Create hook instances
timeline_hook = TimelineHook()
safety_guard = SafetyGuard(blocked_tools=[])  # No blocks for now

# Create the agent with ALL components configured:
agent = Agent(
    # Brain: which AI model to use
    model=BedrockModel(
        model_id="us.anthropic.claude-sonnet-4-20250514",
        max_tokens=1024,
    ),
    
    # Personality: permanent instructions
    system_prompt="You are a helpful assistant. Be concise. Use tools when needed.",
    
    # Hands: tools the agent can use
    tools=[get_weather, calculator, save_note],
    
    # Reflexes: lifecycle hooks
    hooks=[timeline_hook, safety_guard],
    
    # Memory management
    conversation_manager=SlidingWindowConversationManager(),
    
    # Output: disable streaming for cleaner notebook output
    callback_handler=None,
)

print("Agent created with:")
print(f"  Model:                {type(agent.model).__name__}")
print(f"  System prompt:        \"{agent.system_prompt[:50]}...\"")
print(f"  Tools:                {list(agent.tool_registry.registry.keys())}")
print(f"  Hooks:                TimelineHook, SafetyGuard")
print(f"  Conversation manager: {type(agent.conversation_manager).__name__}")
print(f"  Messages:             {len(agent.messages)}")
print(f"  State:                {agent.state}")

In [ ]:
# ============================================================
# STEP 5: Multi-turn conversation (4 calls)
# ============================================================

# Turn 1: Simple question (no tools)
print("=" * 60)
print("TURN 1: Simple question")
print("=" * 60)
result1 = agent("What's your name? Answer briefly.")
print(f"Answer: {result1}")
print(f"Messages: {len(agent.messages)}")
print()

# Turn 2: Requires weather tool
print("=" * 60)
print("TURN 2: Weather check (uses tool)")
print("=" * 60)
result2 = agent("What's the weather in Seattle?")
print(f"Answer: {result2}")
print(f"Messages: {len(agent.messages)}")
print()

# Turn 3: Requires calculator tool
print("=" * 60)
print("TURN 3: Math (uses tool)")
print("=" * 60)
result3 = agent("What is 42 * 17?")
print(f"Answer: {result3}")
print(f"Messages: {len(agent.messages)}")
print()

# Turn 4: Save a note (uses tool with ToolContext)
print("=" * 60)
print("TURN 4: Save note (uses tool with ToolContext)")
print("=" * 60)
result4 = agent("Save a note titled 'Summary' with the content 'Seattle is cloudy, 42*17=714'")
print(f"Answer: {result4}")
print(f"Messages: {len(agent.messages)}")

In [ ]:
# ============================================================
# STEP 6: Full diagnostic printout
# ============================================================

print("=" * 60)
print("FULL AGENT DIAGNOSTIC")
print("=" * 60)
print()

# --- Model ---
print("--- Model ---")
print(f"Type: {type(agent.model).__name__}")
config = agent.model.get_config()
for k, v in config.items():
    print(f"  {k}: {v}")
print()

# --- Tools ---
print("--- Tools ---")
for name, t in agent.tool_registry.registry.items():
    print(f"  {name}: {t.tool_type}")
print()

# --- Messages ---
print("--- Messages ---")
print(f"Total: {len(agent.messages)}")
for i, msg in enumerate(agent.messages):
    types = []
    for block in msg['content']:
        if 'text' in block:
            types.append('text')
        elif 'toolUse' in block:
            types.append(f"toolUse:{block['toolUse']['name']}")
        elif 'toolResult' in block:
            types.append('toolResult')
    print(f"  [{i:2d}] {msg['role']:10s} | {', '.join(types)}")
print()

# --- State ---
print("--- Agent State ---")
print(json.dumps(agent.state, indent=2))
print()

# --- Timeline ---
timeline_hook.print_timeline()

# --- Metrics ---
print("--- Metrics ---")
metrics = agent.event_loop_metrics
print(f"Input tokens:  {metrics.accumulated_usage.get('inputTokens', 'N/A')}")
print(f"Output tokens: {metrics.accumulated_usage.get('outputTokens', 'N/A')}")

---
## Complete Request Flow Trace

Here's what happens for a single tool-using call like `agent("What's the weather in Seattle?")`:

```
1. agent.__call__("What's the weather?")      # You call the agent
2.   invoke_async()                             # Converts to async
3.     stream_async()                           # Acquires lock, prepares prompt
4.       _run_loop()                            # Fires BeforeInvocationEvent
5.         messages.append(user_message)         # Your message added
6.         event_loop_cycle()                    # CYCLE 1 STARTS
7.           BeforeModelCallEvent fired
8.           _handle_model_execution()           # Calls AI model
9.           AfterModelCallEvent fired (stop=tool_use)
10.          MessageAddedEvent fired              # Assistant msg added
11.          _handle_tool_execution()             # 
12.            BeforeToolCallEvent fired (get_weather)
13.            get_weather(city="Seattle") runs   # Your tool code!
14.            AfterToolCallEvent fired
15.          MessageAddedEvent fired              # Tool result added
16.          recurse_event_loop()                 # CYCLE 2 STARTS
17.            BeforeModelCallEvent fired
18.            _handle_model_execution()           # Model sees tool result
19.            AfterModelCallEvent fired (stop=end_turn)
20.            MessageAddedEvent fired              # Final msg added
21.        AfterInvocationEvent fired
22.        conversation_manager.apply()             # Trim if needed
23. AgentResult returned to you
```

---
## Source File Map

| File | What it does | Key classes/functions |
|------|-------------|---------------------|
| `agent/agent.py` | Main Agent class | `Agent`, `__call__`, `stream_async`, `_run_loop` |
| `agent/agent_result.py` | Result object | `AgentResult` |
| `agent/conversation_manager/` | History management | `SlidingWindowConversationManager` |
| `event_loop/event_loop.py` | Core reasoning loop | `event_loop_cycle`, `_handle_model_execution` |
| `models/model.py` | Model interface | `Model` (ABC) |
| `models/bedrock.py` | Amazon Bedrock adapter | `BedrockModel` |
| `tools/decorator.py` | @tool decorator | `tool`, `DecoratorTool` |
| `tools/registry.py` | Tool storage | `ToolRegistry` |
| `hooks/events.py` | Hook event definitions | `BeforeToolCallEvent`, etc. |
| `hooks/registry.py` | Hook dispatch | `HookRegistry`, `HookProvider` |
| `types/content.py` | Message types | `Message`, `ContentBlock` |
| `types/tools.py` | Tool types | `ToolSpec`, `ToolUse`, `ToolResult`, `ToolContext` |
| `types/streaming.py` | Streaming types | `StreamEvent`, `StopReason` |
| `interrupt.py` | Human-in-the-loop | `Interrupt`, `InterruptException` |
| `multiagent/` | Multi-agent patterns | `GraphAgent`, `SwarmAgent` |

---
## Where to Go Next

You now understand the fundamentals of the Strands Agents SDK. Here's what to explore next:

| Topic | What you'll learn | Where |
|-------|------------------|-------|
| **Interrupts** | Human-in-the-loop: pause agent, get user input, resume | `../interrupt/` folder |
| **Multi-Agent** | Orchestrate multiple agents (Graph, Swarm patterns) | `src/strands/multiagent/` |
| **Sessions** | Save and restore conversations across restarts | `src/strands/session/` |
| **Observability** | OpenTelemetry tracing, metrics, logging | `src/strands/telemetry/` |
| **MCP Tools** | Connect to external tool servers (Model Context Protocol) | `src/strands/tools/mcp/` |
| **BIDI Streaming** | Real-time audio/bidirectional streaming | `src/strands/experimental/bidi/` |
| **Custom Model** | Create your own model adapter | Subclass `Model` ABC |

**Congratulations!** You now have a solid foundation in the Strands Agents SDK architecture.